# Package existing Sentinel-1 work for Azure resume

This notebook replaces the WSL Bash packaging script with ordinary Python cells. It creates **one transfer archive** containing only:

- a consistent SQLite backup of `scene_inventory_nearest_v1.sqlite` (not live SQLite sidecars);
- `stations_with_clusters.parquet`, preserving the existing station-to-cluster IDs;
- all existing `.npz` shards from `scene_observations_nearest_overview_v1`.

It does **not** rebuild the inventory, recalculate clusters, delete anything, or include the large STAC caches and old 12-day-median Zarr outputs. The first cells perform a read-only audit. The archive is built only when you explicitly run the final packaging cell.

Run cells from top to bottom. If your local WSL project path differs, edit the path in the configuration cell.


In [1]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import os
import sqlite3
import tarfile
import tempfile
import time

# EDIT THIS if the local WSL project is somewhere else.
PROJECT_ROOT = Path('/home/mayur/Code/UG_SM').expanduser().resolve()

SENTINEL_REL = Path('Data/Global/pipeline_stage1/sentinel1')
SENTINEL_ROOT = PROJECT_ROOT / SENTINEL_REL
INVENTORY_PATH = SENTINEL_ROOT / 'scene_inventory_nearest_v1.sqlite'
CLUSTER_MAPPING_PATH = SENTINEL_ROOT / 'stations_with_clusters.parquet'
SHARDS_DIR = SENTINEL_ROOT / 'scene_observations_nearest_overview_v1'

# The archive is written in the project root so it is easy to find in local VS Code.
STAMP = datetime.now().strftime('%Y%m%d_%H%M%S')
ARCHIVE_PATH = PROJECT_ROOT / f'sentinel1_azure_resume_bundle_{STAMP}.tar.gz'
CHECKSUM_PATH = Path(str(ARCHIVE_PATH) + '.sha256')

print('Project root:', PROJECT_ROOT)
print('Sentinel directory:', SENTINEL_ROOT)
print('Inventory:', INVENTORY_PATH)
print('Cluster mapping:', CLUSTER_MAPPING_PATH)
print('Existing shards:', SHARDS_DIR)
print('Archive target:', ARCHIVE_PATH)


Project root: /home/mayur/Code/UG_SM
Sentinel directory: /home/mayur/Code/UG_SM/Data/Global/pipeline_stage1/sentinel1
Inventory: /home/mayur/Code/UG_SM/Data/Global/pipeline_stage1/sentinel1/scene_inventory_nearest_v1.sqlite
Cluster mapping: /home/mayur/Code/UG_SM/Data/Global/pipeline_stage1/sentinel1/stations_with_clusters.parquet
Existing shards: /home/mayur/Code/UG_SM/Data/Global/pipeline_stage1/sentinel1/scene_observations_nearest_overview_v1
Archive target: /home/mayur/Code/UG_SM/sentinel1_azure_resume_bundle_20261010_133846.tar.gz


In [2]:
# Preflight: this cell reads metadata only; it does not create or modify project data.
required = [PROJECT_ROOT, INVENTORY_PATH, CLUSTER_MAPPING_PATH, SHARDS_DIR]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError('Missing required path(s):\n  ' + '\n  '.join(missing))
if not PROJECT_ROOT.is_dir():
    raise NotADirectoryError(PROJECT_ROOT)
if ARCHIVE_PATH.exists() or CHECKSUM_PATH.exists():
    raise FileExistsError('Archive target already exists; rerun the configuration cell to generate a new timestamp.')

shard_files = sorted(p for p in SHARDS_DIR.iterdir() if p.is_file() and p.suffix == '.npz')
print(f'Inventory size: {INVENTORY_PATH.stat().st_size / (1024**3):.2f} GiB')
print(f'Cluster mapping size: {CLUSTER_MAPPING_PATH.stat().st_size / (1024**2):.2f} MiB')
print(f'Existing NPZ shards: {len(shard_files):,}')
print(f'Shard directory size: {sum(p.stat().st_size for p in shard_files) / (1024**2):.1f} MiB')

# A read-only SQLite check, including consistency of the cluster links.
uri = INVENTORY_PATH.as_uri() + '?mode=ro'
with sqlite3.connect(uri, uri=True, timeout=30) as con:
    tables = {r[0] for r in con.execute("SELECT name FROM sqlite_master WHERE type='table'")}
    expected_tables = {'cluster_searches', 'scene_clusters', 'scenes'}
    if not expected_tables.issubset(tables):
        raise RuntimeError(f'Unexpected inventory schema; tables found: {sorted(tables)}')
    scene_count = con.execute('SELECT COUNT(*) FROM scenes').fetchone()[0]
    completed_clusters = con.execute('SELECT COUNT(*) FROM cluster_searches WHERE completed=1').fetchone()[0]
    total_clusters = con.execute('SELECT COUNT(*) FROM cluster_searches').fetchone()[0]
    cluster_links = con.execute('SELECT COUNT(*) FROM scene_clusters').fetchone()[0]
    linked_cluster_count = con.execute('SELECT COUNT(DISTINCT cluster_id) FROM scene_clusters').fetchone()[0]
    unlinked_scenes = con.execute('''SELECT COUNT(*) FROM scenes s LEFT JOIN scene_clusters sc ON sc.scene_id=s.scene_id WHERE sc.scene_id IS NULL''').fetchone()[0]
    zero_result_clusters = con.execute('SELECT cluster_id FROM cluster_searches WHERE result_count=0 ORDER BY cluster_id').fetchall()

print(f'Unique scenes in inventory: {scene_count:,}')
print(f'Cluster searches marked completed: {completed_clusters:,}/{total_clusters:,}')
print(f'Scene-cluster association rows: {cluster_links:,}')
print(f'Clusters with scene associations: {linked_cluster_count:,}')
print(f'Scenes without any cluster link: {unlinked_scenes:,}')
print('Zero-result clusters:', [r[0] for r in zero_result_clusters])

if completed_clusters != total_clusters:
    raise RuntimeError('Not all cluster searches are marked complete; inspect the inventory before packaging.')
if unlinked_scenes != 0:
    raise RuntimeError('Some inventory scenes have no cluster link; inspect the inventory before packaging.')
if scene_count != 204033:
    print('WARNING: scene count differs from the last audit (204,033). Verify this is the intended inventory.')
if len(shard_files) != 21655:
    print('WARNING: shard count differs from the last audit (21,655). This may be okay if work changed since then.')


Inventory size: 1.36 GiB
Cluster mapping size: 0.12 MiB
Existing NPZ shards: 21,655
Shard directory size: 108.3 MiB
Unique scenes in inventory: 204,033
Cluster searches marked completed: 819/819
Scene-cluster association rows: 528,280
Clusters with scene associations: 818
Scenes without any cluster link: 0
Zero-result clusters: [397]


## Package the resume state

Run the next cell only after reviewing the audit above. It makes a temporary SQLite backup using SQLite's backup API, packages the backup plus the mapping and current NPZ shards, and writes a SHA-256 checksum beside the archive. Compression uses gzip level 1 to reduce CPU time; the NPZ files are already compressed internally. The original inventory, mapping, and shards are not deleted or modified.


In [3]:
# Create the archive. This may take several minutes depending on local disk speed.
started = time.time()
manifest = {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'project_relative_sentinel_path': str(SENTINEL_REL),
    'inventory_filename': INVENTORY_PATH.name,
    'cluster_mapping_filename': CLUSTER_MAPPING_PATH.name,
    'scene_shard_directory': SHARDS_DIR.name,
    'inventory_scene_count': int(scene_count),
    'cluster_searches_completed': int(completed_clusters),
    'cluster_search_count': int(total_clusters),
    'scene_cluster_link_rows': int(cluster_links),
    'clusters_with_scene_links': int(linked_cluster_count),
    'scenes_without_cluster_links': int(unlinked_scenes),
    'existing_npz_shards': int(len(shard_files)),
    'included_stac_caches': False,
    'included_legacy_yearly_zarr': False,
}
manifest_bytes = json.dumps(manifest, indent=2).encode('utf-8')
archive_inventory_arcname = (SENTINEL_REL / INVENTORY_PATH.name).as_posix()
archive_mapping_arcname = (SENTINEL_REL / CLUSTER_MAPPING_PATH.name).as_posix()
archive_shards_arcdir = (SENTINEL_REL / SHARDS_DIR.name).as_posix()

# Use a temporary DB snapshot. SQLite's backup API is designed for a consistent snapshot,
# including committed WAL data; do not archive the live -wal/-shm files separately.
with tempfile.TemporaryDirectory(prefix='sentinel_inventory_backup_') as temp_dir:
    snapshot_path = Path(temp_dir) / INVENTORY_PATH.name
    src = sqlite3.connect(INVENTORY_PATH, timeout=60)
    try:
        src.execute('PRAGMA query_only=ON')
        dst = sqlite3.connect(snapshot_path)
        try:
            src.backup(dst, pages=1000, sleep=0.05)
            dst.commit()
        finally:
            dst.close()
    finally:
        src.close()

    # Check the snapshot before placing it in the bundle.
    with sqlite3.connect(f'file:{snapshot_path.as_posix()}?mode=ro', uri=True, timeout=30) as snap:
        snap_scene_count = snap.execute('SELECT COUNT(*) FROM scenes').fetchone()[0]
        snap_completed = snap.execute('SELECT COUNT(*) FROM cluster_searches WHERE completed=1').fetchone()[0]
    if snap_scene_count != scene_count or snap_completed != completed_clusters:
        raise RuntimeError('SQLite snapshot counts differ from the preflight; archive was not completed.')

    with tarfile.open(ARCHIVE_PATH, mode='w:gz', compresslevel=1) as tar:
        tar.add(snapshot_path, arcname=archive_inventory_arcname, recursive=False)
        tar.add(CLUSTER_MAPPING_PATH, arcname=archive_mapping_arcname, recursive=False)
        for i, p in enumerate(shard_files, start=1):
            tar.add(p, arcname=f'{archive_shards_arcdir}/{p.name}', recursive=False)
            if i % 1000 == 0 or i == len(shard_files):
                print(f'Added {i:,}/{len(shard_files):,} shard files...')
        info = tarfile.TarInfo(name=(SENTINEL_REL / 'azure_resume_manifest.json').as_posix())
        info.size = len(manifest_bytes)
        import io
        tar.addfile(info, io.BytesIO(manifest_bytes))

print('Archive written:', ARCHIVE_PATH)
print('Archive size:', f'{ARCHIVE_PATH.stat().st_size / (1024**3):.2f} GiB')
print('Elapsed minutes:', f'{(time.time() - started) / 60:.1f}')


Added 1,000/21,655 shard files...
Added 2,000/21,655 shard files...
Added 3,000/21,655 shard files...
Added 4,000/21,655 shard files...
Added 5,000/21,655 shard files...
Added 6,000/21,655 shard files...
Added 7,000/21,655 shard files...
Added 8,000/21,655 shard files...
Added 9,000/21,655 shard files...
Added 10,000/21,655 shard files...
Added 11,000/21,655 shard files...
Added 12,000/21,655 shard files...
Added 13,000/21,655 shard files...
Added 14,000/21,655 shard files...
Added 15,000/21,655 shard files...
Added 16,000/21,655 shard files...
Added 17,000/21,655 shard files...
Added 18,000/21,655 shard files...
Added 19,000/21,655 shard files...
Added 20,000/21,655 shard files...
Added 21,000/21,655 shard files...
Added 21,655/21,655 shard files...
Archive written: /home/mayur/Code/UG_SM/sentinel1_azure_resume_bundle_20261010_133846.tar.gz
Archive size: 0.22 GiB
Elapsed minutes: 0.3


In [4]:
# Calculate a SHA-256 checksum without using shell commands.
sha = hashlib.sha256()
with ARCHIVE_PATH.open('rb') as f:
    for block in iter(lambda: f.read(8 * 1024 * 1024), b''):
        sha.update(block)
checksum = sha.hexdigest()
CHECKSUM_PATH.write_text(f'{checksum}  {ARCHIVE_PATH.name}\n', encoding='utf-8')
print('SHA-256:', checksum)
print('Checksum file:', CHECKSUM_PATH)
print('\nTRANSFER THESE FILES TO THE AZURE VS CODE WINDOW:')
print('1.', ARCHIVE_PATH)
print('2.', CHECKSUM_PATH)
print('\nDo not delete the original database or shards. After the archive has been verified and extracted on Azure, the transfer archive can be deleted from the VM to reclaim space.')


SHA-256: 7d76ee8147458a53be31e421a58ba0ee63974d94ba19dc539cdf2a6efa22b803
Checksum file: /home/mayur/Code/UG_SM/sentinel1_azure_resume_bundle_20261010_133846.tar.gz.sha256

TRANSFER THESE FILES TO THE AZURE VS CODE WINDOW:
1. /home/mayur/Code/UG_SM/sentinel1_azure_resume_bundle_20261010_133846.tar.gz
2. /home/mayur/Code/UG_SM/sentinel1_azure_resume_bundle_20261010_133846.tar.gz.sha256

Do not delete the original database or shards. After the archive has been verified and extracted on Azure, the transfer archive can be deleted from the VM to reclaim space.


## Contents and safety notes

The tar archive preserves paths under `Data/Global/pipeline_stage1/sentinel1/`, so extracting it from the Azure project root recreates the expected layout. It includes only the inventory snapshot, saved cluster mapping, existing overview `.npz` files, and a manifest. It excludes STAC caches, duplicate inventory copies, and the older 12-day-median Zarrs.

The archive is a transfer copy; it does not free local disk space, and the Azure copy is not a substitute for keeping the original until the resume run has been verified.
